In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook explores the behavior of a continuous-time
# low-pass Chebyshev Type I filter.
#
# Three parameters can be varied:
#
#       N    : filter order
#       ωp   : passband edge angular frequency
#       Ap   : maximum passband ripple in dB
#
# The ripple parameter is calculated automatically from
#
#       ε = sqrt(10^(Ap/10) - 1).
#
# The filter is constructed directly with scipy.signal.cheby1 using
# analog=True.
#
# The notebook displays four responses:
#
# 1. Magnitude response |H(jω)|
#    The Chebyshev I response exhibits equal-amplitude ripple in the
#    passband and monotonic attenuation in the stopband.
#
#    At ω = ωp:
#
#       |H(jωp)| = 10^(-Ap/20) = 1/sqrt(1 + ε²).
#
#    Increasing N sharpens the transition between the passband and stopband.
#
# 2. Phase response ∠H(jω)
#    Increasing N increases the total phase shift.
#
# 3. Impulse response h(t)
#    The transient response changes with N, ωp and Ap.
#
# 4. Step response
#    For odd-order filters the final value is 1.
#    For even-order filters the final value is
#
#       10^(-Ap/20) = 1/sqrt(1 + ε²).
#
# The time-domain plots use fixed axes so that changes in the curves can be
# observed directly instead of being hidden by automatic axis rescaling.
#
# Increasing ωp produces a faster time-domain response.
#
# IMPORTANT DIFFERENCE FROM BUTTERWORTH:
#
# Butterworth:
#       maximally flat passband, no ripple.
#
# Chebyshev I:
#       equal ripple in the passband and a sharper transition.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 7px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:780px;
    max-width:780px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Explore the frequency-domain and time-domain behavior of a continuous-time low-pass Chebyshev Type I filter.
<br>
<b>Interpretation:</b>
The sliders control the filter order N, passband edge frequency ωp and maximum passband ripple Ap. Increasing N sharpens the transition, while Ap controls the amplitude of the equal ripple in the passband. At ω = ωp, |H(jωp)| = 10<sup>−Ap/20</sup>.
</div>
""", layout=Layout(width='790px', max_width='790px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='250px')
style_opts = {'description_width':'75px'}

order_slider = IntSlider(min=2, max=10, step=1, value=2, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

wp_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='ωp:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

Ap_slider = FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Ap (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='270px', max_width='270px'))

# ==============================================================================
# FIGURE 1: MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(5.2, 3.15))

mag_line, = ax_mag.plot([], [], 'r-', linewidth=2.0, label='|H(jω)|')

wp_mag_line = ax_mag.axvline(1.0, color='black', linestyle=':', linewidth=1.2, label='ωp')

ripple_line = ax_mag.axhline(10.0**(-1.0 / 20.0), color='gray', linestyle='--', linewidth=1.0, label='Ripple limit')

passband_top_line = ax_mag.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)

edge_point, = ax_mag.plot([1.0], [10.0**(-1.0 / 20.0)], 'ko', markersize=4)

ax_mag.set_xscale('log')

ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_mag.set_ylabel('|H(jω)|', fontsize=10)

ax_mag.set_title('Chebyshev I Magnitude Response', fontsize=12, fontweight='bold', pad=5)

ax_mag.tick_params(axis='both', labelsize=9)

ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)

ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=3, fontsize=8)

ax_mag.set_xlim(0.05, 50.0)
ax_mag.set_ylim(0.0, 1.08)

fig_mag.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_mag.canvas.header_visible = False
fig_mag.canvas.toolbar_visible = False
fig_mag.canvas.resizable = False

fig_mag.canvas.layout.width = '520px'
fig_mag.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 2: PHASE RESPONSE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(5.2, 3.15))

phase_line, = ax_phase.plot([], [], 'r-', linewidth=2.0, label='∠H(jω)')

wp_phase_line = ax_phase.axvline(1.0, color='black', linestyle=':', linewidth=1.2, label='ωp')

zero_phase_line = ax_phase.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_phase.set_xscale('log')

ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_phase.set_ylabel('Phase (degrees)', fontsize=10)

ax_phase.set_title('Chebyshev I Phase Response', fontsize=12, fontweight='bold', pad=5)

ax_phase.tick_params(axis='both', labelsize=9)

ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)

ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)

ax_phase.set_xlim(0.05, 50.0)

fig_phase.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_phase.canvas.header_visible = False
fig_phase.canvas.toolbar_visible = False
fig_phase.canvas.resizable = False

fig_phase.canvas.layout.width = '520px'
fig_phase.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 3: IMPULSE RESPONSE
# ==============================================================================

fig_impulse, ax_impulse = plt.subplots(figsize=(5.2, 3.15))

impulse_line, = ax_impulse.plot([], [], 'r-', linewidth=2.0, label='h(t)')

zero_impulse_line = ax_impulse.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_impulse.set_xlabel('Time t (s)', fontsize=10)
ax_impulse.set_ylabel('h(t)', fontsize=10)

ax_impulse.set_title('Chebyshev I Impulse Response', fontsize=12, fontweight='bold', pad=5)

ax_impulse.tick_params(axis='both', labelsize=9)

ax_impulse.grid(True, linestyle=':', alpha=0.5)

ax_impulse.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=1, fontsize=8)

# Fixed time-domain axes
ax_impulse.set_xlim(0.0, 24.0)
ax_impulse.set_xticks([0, 4, 8, 12, 16, 20, 24])

ax_impulse.set_ylim(-1.5, 3.0)
ax_impulse.set_yticks([-1.5, -1.0, -0.5, 0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])

fig_impulse.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_impulse.canvas.header_visible = False
fig_impulse.canvas.toolbar_visible = False
fig_impulse.canvas.resizable = False

fig_impulse.canvas.layout.width = '520px'
fig_impulse.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 4: STEP RESPONSE
# ==============================================================================

fig_step, ax_step = plt.subplots(figsize=(5.2, 3.15))

step_line, = ax_step.plot([], [], 'r-', linewidth=2.0, label='Step response')

final_value_line = ax_step.axhline(1.0, color='gray', linestyle='--', linewidth=1.0, label='Final value')

ax_step.set_xlabel('Time t (s)', fontsize=10)
ax_step.set_ylabel('Amplitude', fontsize=10)

ax_step.set_title('Chebyshev I Step Response', fontsize=12, fontweight='bold', pad=5)

ax_step.tick_params(axis='both', labelsize=9)

ax_step.grid(True, linestyle=':', alpha=0.5)

ax_step.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)

# Fixed axes
ax_step.set_xlim(0.0, 24.0)
ax_step.set_xticks([0, 4, 8, 12, 16, 20, 24])

ax_step.set_ylim(-0.1, 1.5)
ax_step.set_yticks([0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5])

fig_step.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_step.canvas.header_visible = False
fig_step.canvas.toolbar_visible = False
fig_step.canvas.resizable = False

fig_step.canvas.layout.width = '520px'
fig_step.canvas.layout.height = '320px'

# ==============================================================================
# FIXED TIME AXIS
# ==============================================================================

t_max = 24.0
t = np.linspace(0.0, t_max, 3000)

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_chebyshev1(change=None):

    N = order_slider.value
    wp = wp_slider.value
    Ap = Ap_slider.value

    # --------------------------------------------------------------------------
    # Ripple parameter
    # --------------------------------------------------------------------------

    epsilon = np.sqrt(10.0**(Ap / 10.0) - 1.0)

    ripple_min = 10.0**(-Ap / 20.0)

    # --------------------------------------------------------------------------
    # Chebyshev Type I analog transfer function
    # --------------------------------------------------------------------------

    b, a = signal.cheby1(N, Ap, wp, btype='low', analog=True, output='ba')

    # --------------------------------------------------------------------------
    # Frequency response
    # --------------------------------------------------------------------------

    omega = np.logspace(np.log10(0.05), np.log10(50.0), 2500)

    _, H = signal.freqs(b, a, worN=omega)

    magnitude = np.abs(H)

    phase = np.unwrap(np.angle(H))

    phase_deg = np.rad2deg(phase)

    # --------------------------------------------------------------------------
    # Impulse and step responses
    # --------------------------------------------------------------------------

    system = signal.TransferFunction(b, a)

    t_impulse, h = signal.impulse(system, T=t)

    t_step, y_step = signal.step(system, T=t)

    # --------------------------------------------------------------------------
    # DC gain / final step value
    # --------------------------------------------------------------------------

    dc_gain = np.abs(b[-1] / a[-1])

    # --------------------------------------------------------------------------
    # Magnitude response update
    # --------------------------------------------------------------------------

    mag_line.set_data(omega, magnitude)

    wp_mag_line.set_xdata([wp, wp])

    ripple_line.set_ydata([ripple_min, ripple_min])

    edge_point.set_data([wp], [ripple_min])

    ax_mag.set_xlim(0.05, 50.0)

    ax_mag.set_ylim(0.0, 1.08)

    # --------------------------------------------------------------------------
    # Phase response update
    # --------------------------------------------------------------------------

    phase_line.set_data(omega, phase_deg)

    wp_phase_line.set_xdata([wp, wp])

    ax_phase.set_xlim(0.05, 50.0)

    phase_min = -90.0 * N

    ax_phase.set_ylim(1.05 * phase_min, 5.0)

    # --------------------------------------------------------------------------
    # Impulse response update
    # --------------------------------------------------------------------------

    impulse_line.set_data(t_impulse, h)

    ax_impulse.set_xlim(0.0, 24.0)

    ax_impulse.set_xticks([0, 4, 8, 12, 16, 20, 24])

    ax_impulse.set_ylim(-1.5, 3.0)

    ax_impulse.set_yticks([-1.5, -1.0, -0.5, 0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])

    # --------------------------------------------------------------------------
    # Step response update
    # --------------------------------------------------------------------------

    step_line.set_data(t_step, y_step)

    final_value_line.set_ydata([dc_gain, dc_gain])

    ax_step.set_xlim(0.0, 24.0)

    ax_step.set_xticks([0, 4, 8, 12, 16, 20, 24])

    ax_step.set_ylim(-0.1, 1.5)

    ax_step.set_yticks([0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5])

    # --------------------------------------------------------------------------
    # Poles
    # --------------------------------------------------------------------------

    poles = np.roots(a)

    pole_text = '<br>'.join([f'p{k + 1} = {p.real:+.4f} {p.imag:+.4f}j' for k, p in enumerate(poles)])

    # --------------------------------------------------------------------------
    # Odd / even order behavior
    # --------------------------------------------------------------------------

    if N % 2 == 0:
        dc_behavior = f'Even order → G(0) = {ripple_min:.4f}'
    else:
        dc_behavior = 'Odd order → G(0) = 1'

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 9px;
        margin-top:9px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:265px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Chebyshev Type I low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Passband edge:</b>
        <span style="color:#0066cc;">ωp = {wp:.2f} rad/s</span>
    </div>

    <div>
        <b>Passband ripple:</b>
        <span style="color:#0066cc;">Ap = {Ap:.2f} dB</span>
    </div>

    <div>
        <b>Ripple parameter ε:</b>
        <span style="color:#0066cc;">{epsilon:.5f}</span>
    </div>

    <div>
        <b>|H(jωp)|:</b>
        <span style="color:#0066cc;">{ripple_min:.4f}</span>
    </div>

    <div>
        <b>Asymptotic slope:</b>
        <span style="color:#0066cc;">−{20 * N} dB/decade</span>
    </div>

    <div>
        <b>Final phase:</b>
        <span style="color:#0066cc;">−{90 * N}°</span>
    </div>

    <div>
        <b>DC gain:</b>
        <span style="color:#0066cc;">{dc_gain:.4f}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Order behavior:</b><br>
        <span style="color:#0066cc;">
        {dc_behavior}
        </span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Poles:</b><br>
        <span style="color:#0066cc;">
        {pole_text}
        </span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        Increasing N sharpens the transition. Increasing Ap permits larger
        passband ripple and changes the pole locations and transient response.
        Increasing ωp produces a faster time-domain response.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # REDRAW EXISTING FIGURES ONLY
    # --------------------------------------------------------------------------

    fig_mag.canvas.draw_idle()

    fig_phase.canvas.draw_idle()

    fig_impulse.canvas.draw_idle()

    fig_step.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_chebyshev1, names='value')

wp_slider.observe(update_chebyshev1, names='value')

Ap_slider.observe(update_chebyshev1, names='value')

# ==============================================================================
# LAYOUT: 2 x 2 FIGURE GRID
# ==============================================================================

controls = VBox([parameter_title, order_slider, wp_slider, Ap_slider, info_html], layout=Layout(width='280px', min_width='280px', max_width='280px', flex='0 0 280px', align_items='flex-start'))

top_row = HBox([fig_mag.canvas, fig_phase.canvas], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

bottom_row = HBox([fig_impulse.canvas, fig_step.canvas], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

plot_grid = VBox([top_row, bottom_row], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

main_layout = HBox([controls, plot_grid], layout=Layout(width='1330px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE DATA
# ==============================================================================

update_chebyshev1()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)